# CTD Python Data Pipeline Capstone

## Dataset Choice: Global Superstore

For my data pipeline capstone project, I am choosing the **Global Superstore** dataset. This dataset is a good fit for the rubric because it includes sales, profit, discount, order dates, shipping dates, customer data, product categories, and geography. These columns make it possible to clean real business data, engineer useful features, aggregate by business segments, and create charts that support clear business insights.

This first version focuses on loading the dataset, checking data quality, cleaning common issues, engineering useful columns, and creating early aggregations and visualizations.


## Project Questions

I will use this project to investigate:

1. Which product categories and sub-categories are the most and least profitable?
2. Which regions, markets, or countries create the strongest sales and profit?
3. How do discounts affect profitability and gross margin?
4. Which customers contribute the most lifetime value?


In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="Set2")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)


def list_input_files():
    for dirname, _, filenames in os.walk("/kaggle/input"):
        for filename in filenames:
            print(os.path.join(dirname, filename))


def clean_column_names(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.replace(" ", "_", regex=False)
        .str.replace("-", "_", regex=False)
        .str.replace("/", "_", regex=False)
    )
    return df


def find_global_superstore_file():
    expected_columns = {"Sales", "Profit", "Category"}
    candidates = []

    for path in Path("/kaggle/input").rglob("*"):
        if path.suffix.lower() not in [".csv", ".xls", ".xlsx"]:
            continue

        try:
            if path.suffix.lower() == ".csv":
                preview = pd.read_csv(path, nrows=5, encoding_errors="ignore")
            else:
                preview = pd.read_excel(path, nrows=5)
        except Exception:
            continue

        if expected_columns.issubset(set(preview.columns)):
            score = sum(word in str(path).lower() for word in ["global", "superstore", "sales"])
            candidates.append((score, path))

    if not candidates:
        raise FileNotFoundError(
            "Could not find a Global Superstore file. Add the Global Superstore dataset as a Kaggle input, then rerun."
        )

    candidates.sort(reverse=True)
    return candidates[0][1]


def load_table(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, encoding_errors="ignore")
    return pd.read_excel(path)


list_input_files()


## Load The Dataset

The first step is to locate the Global Superstore file from the Kaggle input folder and load it into a Pandas DataFrame. I use a helper function so the notebook is more reproducible even if Kaggle places the dataset in a slightly different folder name.


In [ ]:
data_path = find_global_superstore_file()
print(f"Loading data from: {data_path}")

raw_df = load_table(data_path)
df = clean_column_names(raw_df)

print("First 5 rows")
display(df.head())

print("Last 5 rows")
display(df.tail())

print("Dataset info")
df.info()


## Initial Data Quality Review

Here I check the shape of the dataset, duplicate rows, missing values, and column types. These checks help me decide which cleaning steps are necessary before analysis.


In [ ]:
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")
print(f"Duplicate rows: {df.duplicated().sum()}")

missing_summary = (
    df.isna()
    .sum()
    .to_frame("missing_count")
    .assign(missing_percent=lambda table: table["missing_count"] / len(df) * 100)
    .sort_values("missing_count", ascending=False)
)

display(missing_summary.head(20))
display(df.describe(include="all").T.head(25))


## Data Cleaning

Cleaning steps performed:

- Standardized column names by replacing spaces with underscores.
- Removed duplicate rows if any existed.
- Stripped whitespace from text columns.
- Converted date columns to datetime values.
- Converted key financial columns such as `Sales`, `Profit`, `Quantity`, `Discount`, and `Shipping_Cost` to numeric values when present.
- Filled missing `Postal_Code` values with `"Unknown"` because missing postal codes do not prevent sales/profit analysis.
- Dropped rows missing critical financial fields because those rows cannot support the main project questions.


In [ ]:
df_clean = df.drop_duplicates().copy()

text_columns = df_clean.select_dtypes(include="object").columns
for column in text_columns:
    df_clean[column] = df_clean[column].astype(str).str.strip()

date_columns = [column for column in ["Order_Date", "Ship_Date"] if column in df_clean.columns]
for column in date_columns:
    df_clean[column] = pd.to_datetime(df_clean[column], errors="coerce")

numeric_columns = [
    column
    for column in ["Sales", "Profit", "Quantity", "Discount", "Shipping_Cost"]
    if column in df_clean.columns
]
for column in numeric_columns:
    df_clean[column] = (
        df_clean[column]
        .astype(str)
        .str.replace("$", "", regex=False)
        .str.replace(",", "", regex=False)
    )
    df_clean[column] = pd.to_numeric(df_clean[column], errors="coerce")

if "Postal_Code" in df_clean.columns:
    df_clean["Postal_Code"] = df_clean["Postal_Code"].replace("nan", np.nan).fillna("Unknown")

critical_columns = [column for column in ["Sales", "Profit", "Order_Date"] if column in df_clean.columns]
df_clean = df_clean.dropna(subset=critical_columns)

print("Cleaned dataset info")
df_clean.info()

print("Remaining missing values in important columns")
display(df_clean[critical_columns].isna().sum())

display(df_clean.head())


## Feature Engineering

I created new columns that will help with analysis:

- `Order_Year`: year extracted from the order date.
- `Order_Month`: month extracted from the order date.
- `Gross_Margin`: profit divided by sales.
- `Discount_Bucket`: a categorical version of discount level.
- `Shipping_Days`: number of days between order date and ship date.
- `Profitable_Order`: whether the order produced positive profit.


In [ ]:
df_project = df_clean.copy()

if "Order_Date" in df_project.columns:
    df_project["Order_Year"] = df_project["Order_Date"].dt.year
    df_project["Order_Month"] = df_project["Order_Date"].dt.month
    df_project["Order_Month_Name"] = df_project["Order_Date"].dt.month_name()

if "Ship_Date" in df_project.columns and "Order_Date" in df_project.columns:
    df_project["Shipping_Days"] = (df_project["Ship_Date"] - df_project["Order_Date"]).dt.days

df_project["Gross_Margin"] = np.where(
    df_project["Sales"] != 0,
    df_project["Profit"] / df_project["Sales"],
    np.nan,
)

if "Discount" in df_project.columns:
    df_project["Discount_Bucket"] = pd.cut(
        df_project["Discount"],
        bins=[-0.01, 0, 0.10, 0.30, 1],
        labels=["None", "Low", "Medium", "High"],
    )

df_project["Profitable_Order"] = df_project["Profit"].map(lambda profit: "Yes" if profit > 0 else "No")

display(df_project.head())


## Aggregation 1: Profit By Category And Sub-Category

This aggregation identifies which product groups are producing the most profit and which ones may need attention.


In [ ]:
subcategory_column = "Sub_Category" if "Sub_Category" in df_project.columns else "Sub-Category"

category_profit = (
    df_project
    .groupby(["Category", subcategory_column], dropna=False)
    .agg(
        total_sales=("Sales", "sum"),
        total_profit=("Profit", "sum"),
        average_margin=("Gross_Margin", "mean"),
        order_count=("Profit", "count"),
    )
    .reset_index()
    .sort_values("total_profit", ascending=False)
)

display(category_profit.head(10))
display(category_profit.tail(10))


## Aggregation 2: Sales And Profit By Region Or Market

This aggregation compares business performance across geography. The code uses `Market` if it exists, otherwise it falls back to `Region`.


In [ ]:
geo_column = "Market" if "Market" in df_project.columns else "Region"

geo_performance = (
    df_project
    .groupby(geo_column, dropna=False)
    .agg(
        total_sales=("Sales", "sum"),
        total_profit=("Profit", "sum"),
        average_margin=("Gross_Margin", "mean"),
        order_count=("Profit", "count"),
    )
    .reset_index()
    .sort_values("total_profit", ascending=False)
)

display(geo_performance)


## Aggregation 3: Customer Lifetime Value

For this project, customer lifetime value is calculated as the total profit generated by each customer across all orders.


In [ ]:
customer_column = "Customer_Name" if "Customer_Name" in df_project.columns else "Customer_ID"

customer_lifetime_value = (
    df_project
    .groupby(customer_column, dropna=False)
    .agg(
        total_sales=("Sales", "sum"),
        total_profit=("Profit", "sum"),
        order_count=("Profit", "count"),
        average_margin=("Gross_Margin", "mean"),
    )
    .reset_index()
    .sort_values("total_profit", ascending=False)
)

display(customer_lifetime_value.head(10))


## Aggregation 4: Discount Impact

This aggregation looks at whether deeper discounts are associated with stronger or weaker profit margins.


In [ ]:
if "Discount_Bucket" in df_project.columns:
    discount_summary = (
        df_project
        .groupby("Discount_Bucket", observed=False)
        .agg(
            total_sales=("Sales", "sum"),
            total_profit=("Profit", "sum"),
            average_margin=("Gross_Margin", "mean"),
            average_discount=("Discount", "mean"),
            order_count=("Profit", "count"),
        )
        .reset_index()
    )
    display(discount_summary)
else:
    print("Discount column was not available in this dataset.")


## Visualization 1: Top 10 Sub-Categories By Profit

This chart shows which sub-categories contribute the most profit. These are the strongest product groups by total profit.


In [ ]:
top_subcategories = category_profit.head(10).sort_values("total_profit")

plt.figure(figsize=(10, 6))
sns.barplot(data=top_subcategories, x="total_profit", y=subcategory_column, hue="Category", dodge=False)
plt.title("Top 10 Sub-Categories By Total Profit")
plt.xlabel("Total Profit")
plt.ylabel("Sub-Category")
plt.legend(title="Category", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.show()


## Visualization 2: Sales And Profit By Geography

This chart compares total sales and total profit by market or region. A geography can have high sales but lower profit, so both metrics are important.


In [ ]:
geo_long = geo_performance.melt(
    id_vars=geo_column,
    value_vars=["total_sales", "total_profit"],
    var_name="Metric",
    value_name="Amount",
)

plt.figure(figsize=(11, 6))
sns.barplot(data=geo_long, x=geo_column, y="Amount", hue="Metric")
plt.title(f"Sales And Profit By {geo_column}")
plt.xlabel(geo_column.replace("_", " "))
plt.ylabel("Amount")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()


## Visualization 3: Discount Bucket Versus Gross Margin

This chart helps test whether larger discounts are connected to lower profit margins.


In [ ]:
if "Discount_Bucket" in df_project.columns:
    plt.figure(figsize=(9, 6))
    sns.boxplot(data=df_project, x="Discount_Bucket", y="Gross_Margin")
    plt.title("Gross Margin Distribution By Discount Bucket")
    plt.xlabel("Discount Bucket")
    plt.ylabel("Gross Margin")
    plt.ylim(-2, 2)
    plt.tight_layout()
    plt.show()
else:
    print("Discount bucket was not created because Discount was unavailable.")


## Visualization 4: Profit Trend Over Time

This line chart shows whether profit is improving or declining over time.


In [ ]:
if "Order_Year" in df_project.columns:
    yearly_profit = (
        df_project
        .groupby("Order_Year")
        .agg(total_sales=("Sales", "sum"), total_profit=("Profit", "sum"))
        .reset_index()
    )

    plt.figure(figsize=(10, 5))
    sns.lineplot(data=yearly_profit, x="Order_Year", y="total_profit", marker="o", label="Profit")
    sns.lineplot(data=yearly_profit, x="Order_Year", y="total_sales", marker="o", label="Sales")
    plt.title("Sales And Profit Trend By Year")
    plt.xlabel("Order Year")
    plt.ylabel("Amount")
    plt.tight_layout()
    plt.show()

    display(yearly_profit)
else:
    print("Order_Year was not created because Order_Date was unavailable.")


## Early Insights

1. **Product profitability is uneven.** The top and bottom sub-category tables show that some product lines produce much stronger profit than others. This will help identify which categories deserve more attention in the final analysis.

2. **Sales and profit should be analyzed together.** A region or market can generate high sales but still have weaker profit, which may indicate higher discounts, shipping costs, or less profitable product mix.

3. **Discounting may reduce margin.** The discount bucket analysis and boxplot help show whether higher discounts are associated with weaker gross margin.

These are early findings. In the final version, I will refine the visualizations, add more written interpretation, and choose the strongest three insights to present.


## Limitations And Next Steps

Current limitations:

- The project depends on the Kaggle version of Global Superstore that is added as an input.
- Some columns may vary slightly between dataset versions, so the notebook uses fallback logic where possible.
- Early charts show patterns, but final conclusions should compare multiple metrics before making recommendations.

Next steps:

- Add more detailed analysis of low-profit products.
- Compare discount levels by category and region.
- Investigate whether shipping time or shipping cost affects profit.
- Polish the final three insights with stronger markdown explanations.
